In [ ]:
import sys, warnings, tempfile
warnings.filterwarnings('ignore', category=FutureWarning)
try:
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')
except AttributeError:
    pass

import time
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPyImage

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from gaffer import config
from gaffer.detection.detector import Detection, FootballDetector
from gaffer.detection.team_assigner import TeamAssigner
from gaffer.tracking.tracker import PlayerTracker

TMP = Path(tempfile.gettempdir())

def show_fig(fname):
    plt.savefig(str(fname), dpi=100, bbox_inches='tight')
    plt.close()
    display(IPyImage(str(fname)))

print('Day 3 — ByteTrack integration')

In [ ]:
CLIP = ROOT / 'data' / 'test_clips' / 'tactical_playlist_1.mp4'
assert CLIP.exists(), f'Missing: {CLIP}'

cap = cv2.VideoCapture(str(CLIP))
FPS   = cap.get(cv2.CAP_PROP_FPS) or 25.0
W     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

# Process 300 frames starting at the action section (same window as Day 2)
ACTION_START = int(30 * FPS)   # skip pre-match intro
N_PROCESS    = 300

print(f'Clip: {CLIP.name}')
print(f'Resolution: {W}x{H} @ {FPS:.1f} fps  |  total {TOTAL} frames')
print(f'Window: frames {ACTION_START}–{ACTION_START + N_PROCESS} ({ACTION_START/FPS:.1f}s–{(ACTION_START+N_PROCESS)/FPS:.1f}s)')

In [ ]:
print('Loading YOLO + warming up ...')
t0 = time.time()
detector = FootballDetector(verbose=False)
print(f'  Model: {detector.model_type}  |  {time.time()-t0:.1f}s warm-up')

assigner = TeamAssigner()
tracker  = PlayerTracker(fps=FPS)
print(f'  TeamAssigner: k={assigner.n_clusters}')
print(f'  PlayerTracker: thresh={tracker._tracker.track_activation_threshold}, buffer={tracker._tracker.maximum_frames_without_update}')
print(f'  detect_every_n={config.DETECT_EVERY_N_FRAMES}, track_ids carry forward on cache frames')

In [ ]:
# Fit TeamAssigner on 8 frames sampled from the action window
cap = cv2.VideoCapture(str(CLIP))
sample_indices = np.linspace(ACTION_START, ACTION_START + N_PROCESS - 1, 8, dtype=int)
fit_frames, fit_detections = [], []

for idx in sample_indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    if not ret:
        continue
    dets = detector.detect_raw(frame)
    fit_frames.append(frame)
    fit_detections.append(dets)
cap.release()

assigner.fit(fit_frames, fit_detections)

n_crops = assigner.n_fit_samples
c2t = assigner.cluster_to_team
print(f'TeamAssigner fitted on {len(fit_frames)} frames, {n_crops} jersey crops')
print(f'Cluster→team map: {c2t}')

In [ ]:
# --- Main tracking pipeline ---
per_frame_track_ids   = []
per_frame_t0_count    = []
per_frame_t1_count    = []
track_id_frames       = defaultdict(list)
team_id_by_track      = {}
team_votes            = defaultdict(lambda: defaultdict(int))

cap     = cv2.VideoCapture(str(CLIP))
cap.set(cv2.CAP_PROP_POS_FRAMES, ACTION_START)
timings      = []
last_tracked = []   # carry last-tracked detections to cache frames

for local_idx in range(N_PROCESS):
    ret, frame = cap.read()
    if not ret:
        break
    global_idx = ACTION_START + local_idx
    t0 = time.time()

    dets = detector.detect(frame, global_idx)
    dets = assigner.assign(frame, dets)

    if global_idx == detector._last_detect_idx:
        # Detection ran: update tracker, save result
        last_tracked = tracker.update(dets)
        dets = last_tracked
    else:
        # Cache frame: positions unchanged — carry forward track_ids from last update
        dets = last_tracked

    timings.append(time.time() - t0)

    active_ids = {d.track_id for d in dets if d.track_id >= 0}
    per_frame_track_ids.append(active_ids)
    per_frame_t0_count.append(sum(1 for d in dets if d.team_id == 0))
    per_frame_t1_count.append(sum(1 for d in dets if d.team_id == 1))

    for d in dets:
        if d.track_id >= 0:
            track_id_frames[d.track_id].append(local_idx)
            if d.team_id >= 0:
                team_votes[d.track_id][d.team_id] += 1

cap.release()

for tid, votes in team_votes.items():
    team_id_by_track[tid] = max(votes, key=votes.get)

print(f'Processed {N_PROCESS} frames in {sum(timings)*1000:.0f}ms  ({np.mean(timings)*1000:.1f}ms/frame mean)')

In [ ]:
# --- Track stability metrics ---
n_unique_tracks = len(track_id_frames)
track_lengths   = {tid: len(frames) for tid, frames in track_id_frames.items()}
mean_len        = np.mean(list(track_lengths.values())) if track_lengths else 0
max_len         = max(track_lengths.values()) if track_lengths else 0
long_tracks     = sum(1 for l in track_lengths.values() if l >= 100)

active_counts = [len(ids) for ids in per_frame_track_ids]
mean_active   = np.mean(active_counts)
std_active    = np.std(active_counts)

fragmented = 0
for tid, frames in track_id_frames.items():
    frames_sorted = sorted(frames)
    gaps = [frames_sorted[i+1] - frames_sorted[i] for i in range(len(frames_sorted)-1)]
    if any(g > 6 for g in gaps):
        fragmented += 1

print('=== Track Stability Report ===')
print(f'  Unique track IDs assigned : {n_unique_tracks}')
print(f'  Mean track length         : {mean_len:.1f} frames')
print(f'  Longest track             : {max_len} frames')
print(f'  Tracks >=100 frames long  : {long_tracks}')
print(f'  Fragmented tracks (gaps>6): {fragmented}')
print(f'  Mean active tracks/frame  : {mean_active:.1f} +/- {std_active:.1f}')
print()

t0_tracks = sum(1 for t in team_id_by_track.values() if t == 0)
t1_tracks = sum(1 for t in team_id_by_track.values() if t == 1)
print(f'  T0 tracks: {t0_tracks}  |  T1 tracks: {t1_tracks}')

# PASS criteria for v0.1 (COCO model, no fine-tuning):
#   - not too many IDs (camera sees ~10-16 players; <30 is sane)
#   - long average track (>30 frames = persistent, not flickering)
#   - low fragmentation (< 40% of tracks have gaps > 2 detection cycles)
verdict = 'PASS' if (n_unique_tracks <= 30 and
                     mean_len >= 30 and
                     fragmented <= n_unique_tracks * 0.40) else 'WARN'
print(f'\nVERDICT: {verdict}')

In [ ]:
# --- Plot 1: active track count per frame ---
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

axes[0].plot(active_counts, lw=1, color='steelblue')
axes[0].axhline(mean_active, color='orange', linestyle='--', label=f'mean={mean_active:.1f}')
axes[0].set_title('Active track count per frame')
axes[0].set_xlabel('Frame (local)')
axes[0].set_ylabel('# tracks')
axes[0].legend()
axes[0].set_ylim(0, 30)

axes[1].plot(per_frame_t0_count, lw=1, color='red', label='T0')
axes[1].plot(per_frame_t1_count, lw=1, color='blue', label='T1')
axes[1].set_title('Per-frame team count')
axes[1].set_xlabel('Frame (local)')
axes[1].set_ylabel('# players')
axes[1].legend()
axes[1].set_ylim(0, 15)

plt.tight_layout()
show_fig(TMP / 'd3_counts.png')

In [ ]:
# --- Plot 2: Track timeline (Gantt-style) ---
fig, ax = plt.subplots(figsize=(12, max(4, n_unique_tracks * 0.35)))

sorted_tracks = sorted(track_id_frames.items(), key=lambda x: min(x[1]))
colors = {0: 'red', 1: 'blue', -1: 'grey'}

for row_idx, (tid, frames) in enumerate(sorted_tracks):
    team = team_id_by_track.get(tid, -1)
    c = colors.get(team, 'grey')
    ax.scatter(frames, [row_idx]*len(frames), s=3, c=c, alpha=0.7)
    ax.text(-3, row_idx, str(tid), fontsize=6, va='center', ha='right')

ax.set_xlabel('Frame (local)')
ax.set_ylabel('Track ID (sorted by start)')
ax.set_title(f'Track timelines  ({n_unique_tracks} unique IDs  |  red=T0, blue=T1, grey=unassigned)')
ax.set_xlim(-5, N_PROCESS)
ax.set_ylim(-1, len(sorted_tracks))
plt.tight_layout()
show_fig(TMP / 'd3_timelines.png')

In [ ]:
# --- Produce annotated MP4 with track IDs ---
OUTPUT = ROOT / 'outputs' / 'day3_tracking_demo.mp4'
OUTPUT.parent.mkdir(exist_ok=True)

PALETTE = [
    (255,0,0),(0,0,255),(0,200,0),(200,0,200),(0,200,200),
    (200,200,0),(128,0,128),(0,128,128),(128,128,0),(64,0,128),
    (0,64,128),(128,64,0),(200,100,0),(0,100,200),(100,0,200),
    (180,80,80),(80,180,80),(80,80,180),(180,180,80),(80,180,180),
    (180,80,180),(255,128,0),(0,255,128),(128,0,255),(255,0,128),
    (0,128,255),(128,255,0),(255,64,64),(64,255,64),(64,64,255),
]
TEAM_COLORS = {0: (0,0,220), 1: (220,0,0)}
BALL_COLOR  = (0,255,255)
UNKNOWN     = (0,200,200)

def track_color(track_id):
    return PALETTE[track_id % len(PALETTE)] if track_id >= 0 else UNKNOWN

def annotate(frame, dets):
    out = frame.copy()
    for d in dets:
        x1,y1,x2,y2 = d.bbox
        if d.class_name == 'ball':
            cv2.rectangle(out, (x1,y1),(x2,y2), BALL_COLOR, 2)
            cv2.putText(out, f'ball {d.confidence:.2f}', (x1,y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, BALL_COLOR, 1)
            continue
        box_c = TEAM_COLORS.get(d.team_id, UNKNOWN)
        cv2.rectangle(out, (x1,y1),(x2,y2), box_c, 2)
        tid_c    = track_color(d.track_id)
        lbl      = f'#{d.track_id}' if d.track_id >= 0 else '?'
        team_lbl = f'T{d.team_id}' if d.team_id >= 0 else 'ref'
        cv2.putText(out, f'{lbl} {team_lbl}', (x1, y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, tid_c, 1)
    return out

def burn_hud(frame, local_idx, n_active, ms):
    ts = (ACTION_START + local_idx) / FPS
    for i, l in enumerate([f'Frame {ACTION_START+local_idx}  t={ts:.1f}s',
                            f'Active tracks: {n_active}',
                            f'{ms:.1f}ms/frame']):
        cv2.putText(frame, l, (10, 20 + i*20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 1, cv2.LINE_AA)
    return frame

print('Writing annotated video ...')
fourcc  = cv2.VideoWriter_fourcc(*'mp4v')
writer  = cv2.VideoWriter(str(OUTPUT), fourcc, FPS, (W, H))
cap     = cv2.VideoCapture(str(CLIP))
cap.set(cv2.CAP_PROP_POS_FRAMES, ACTION_START)

vid_tracker   = PlayerTracker(fps=FPS)
vid_last      = []
sample_frames = {0, 50, 100, 150, 200, 250, 299}
saved_frames  = {}

for local_idx in range(N_PROCESS):
    ret, frame = cap.read()
    if not ret:
        break
    global_idx = ACTION_START + local_idx
    t0 = time.time()

    dets = detector.detect(frame, global_idx)
    dets = assigner.assign(frame, dets)

    if global_idx == detector._last_detect_idx:
        vid_last = vid_tracker.update(dets)
        dets = vid_last
    else:
        dets = vid_last   # carry forward track_ids on cache frames

    ms       = (time.time() - t0) * 1000
    n_active = sum(1 for d in dets if d.track_id >= 0)

    out = annotate(frame, dets)
    out = burn_hud(out, local_idx, n_active, ms)
    writer.write(out)

    if local_idx in sample_frames:
        saved_frames[local_idx] = out.copy()

cap.release()
writer.release()

size_mb = OUTPUT.stat().st_size / 1024**2
print(f'Saved: {OUTPUT.name}  ({size_mb:.1f} MB)')

In [ ]:
# --- Display sample frames from the video ---
keys = sorted(saved_frames.keys())
rows = (len(keys) + 2) // 3
fig, axes = plt.subplots(rows, 3, figsize=(18, rows * 4))
axes = axes.flatten()

for i, k in enumerate(keys):
    rgb = cv2.cvtColor(saved_frames[k], cv2.COLOR_BGR2RGB)
    axes[i].imshow(rgb)
    axes[i].set_title(f'Local frame {k}')
    axes[i].axis('off')
for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
show_fig(TMP / 'd3_samples.png')

In [ ]:
# --- Final summary ---
print('=== Day 3 Summary ===')
print(f'  Clip          : {CLIP.name}')
print(f'  Frames        : {N_PROCESS}')
print(f'  Speed         : {np.mean(timings)*1000:.1f} ms/frame mean')
print()
print('  Tracking:')
print(f'    Unique IDs      : {n_unique_tracks}')
print(f'    Mean length     : {mean_len:.1f} frames')
print(f'    Long tracks(≥100): {long_tracks}')
print(f'    Fragmented      : {fragmented}')
print(f'    T0 tracks       : {t0_tracks}')
print(f'    T1 tracks       : {t1_tracks}')
print()
print(f'  VERDICT: {verdict}')
print(f'  Output  : outputs/day3_tracking_demo.mp4  ({OUTPUT.stat().st_size/1024**2:.1f} MB)')